按 PRICE_DATE 重新计算交易日偏股型基金的 return

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

data_path = Path("交易日偏股型基金.feather")
nav = pd.read_feather(data_path)

required_cols = {"F_INFO_WINDCODE", "PRICE_DATE", "F_NAV_ADJUSTED"}
missing_cols = required_cols.difference(nav.columns)
if missing_cols:
    raise KeyError(f"缺少计算 return 所需列: {sorted(missing_cols)}")

# 保存结构信息。后续只允许覆盖 return，不改变其他列、行数或原始行序。
original_columns = nav.columns.tolist()
original_row_count = len(nav)

# 在精简临时表中按 PRICE_DATE 计算，避免排序或清洗操作影响 nav 的其他列。
return_work = nav[["F_INFO_WINDCODE", "PRICE_DATE", "F_NAV_ADJUSTED"]].copy()
return_work["_row_position"] = np.arange(len(return_work))
return_work["_price_date_parsed"] = pd.to_datetime(
    return_work["PRICE_DATE"], errors="coerce"
)
return_work["_nav_for_return"] = pd.to_numeric(
    return_work["F_NAV_ADJUSTED"], errors="coerce"
).replace(0, np.nan)

return_work = return_work.sort_values(
    ["F_INFO_WINDCODE", "_price_date_parsed", "_row_position"],
    kind="stable",
)
return_work["_new_return"] = (
    return_work.groupby("F_INFO_WINDCODE", sort=False)["_nav_for_return"]
    .pct_change(fill_method=None)
    .replace([np.inf, -np.inf], np.nan)
)

# 恢复原始行序，并且只覆盖 return 列。首条记录或净值缺失导致的收益保持 NaN。
new_return = (
    return_work.sort_values("_row_position")["_new_return"]
    .reset_index(drop=True)
)
nav["return"] = new_return.to_numpy()

assert len(nav) == original_row_count
assert nav.columns.tolist() == original_columns


In [2]:
nav[["F_INFO_WINDCODE", "PRICE_DATE", "F_NAV_ADJUSTED", "return"]].head()

,F_INFO_WINDCODE,PRICE_DATE,F_NAV_ADJUSTED,return
0,000001.OF,2002-01-04,1.000,NaN
1,000001.OF,2002-01-11,1.001,0.001000
2,000001.OF,2002-01-18,1.000,-0.000999
3,000001.OF,2002-01-25,1.005,0.005000
4,000001.OF,2002-01-30,1.002,-0.002985


检测收益率计算情况如何

In [3]:
# 查看 return 描述统计
print(nav["return"].describe())

# 查看收益率最大的几行
print(
    nav.sort_values("return", ascending=False)
       [["F_INFO_WINDCODE", "PRICE_DATE", "F_NAV_ADJUSTED", "return"]]
       .head()
)

# 查看收益率最小的几行
print(
    nav.sort_values("return")
       [["F_INFO_WINDCODE", "PRICE_DATE", "F_NAV_ADJUSTED", "return"]]
       .head()
)

# 检查每只基金有多少有效 return
valid_count = (
    nav.groupby("F_INFO_WINDCODE")["return"]
       .count()
)

print(valid_count.describe())

# 查看有效收益率最少的基金
print(valid_count.sort_values().head())

count    1.325655e+07
mean     4.130376e-04
std      1.422016e-02
min     -5.267375e-01
25%     -5.914746e-03
50%      0.000000e+00
75%      6.668572e-03
max      9.056604e-01
Name: return, dtype: float64
         F_INFO_WINDCODE PRICE_DATE  F_NAV_ADJUSTED    return
2231736        002059.OF 2016-04-18        2.424000  0.905660
1456257        001493.OF 2015-07-07        1.711000  0.717871
13240465       970041.OF 2017-05-16        1.308387  0.711876
2936137        002779.OF 2016-11-17        1.624000  0.472348
1496701        001518.OF 2015-10-09        1.495300  0.451466
         F_INFO_WINDCODE PRICE_DATE  F_NAV_ADJUSTED    return
10601549       150072.SZ 2015-03-30        1.996600 -0.526737
10582430       150011.SZ 2013-02-18        0.297200 -0.404409
12384022       519621.OF 2025-03-18        1.035494 -0.338055
12369881       519615.OF 2023-04-27        1.064971 -0.335460
12411191       519646.OF 2022-12-09        1.052364 -0.272198
count    10183.000000
mean      1301.831582
std    

检测有效收益率为0的基金有多少

In [4]:
# 检查 adjusted nav 恒定为 1 的基金

bad_funds = (
    nav.groupby("F_INFO_WINDCODE")["F_NAV_ADJUSTED"]
       .apply(lambda x: x.dropna().nunique() == 1 and x.dropna().iloc[0] == 1)
)

# 这些基金代码
bad_funds = bad_funds[bad_funds].index.tolist()

print("加权净值异常基金数量:", len(bad_funds))
print("没有有效收益率的基金数量:", (valid_count == 0).sum())

加权净值异常基金数量: 18
没有有效收益率的基金数量: 17


保存数据

In [5]:
import os
import pyarrow.dataset as ds

# 先写同目录临时文件并检查结构，成功后再原子替换正式文件。
temp_path = data_path.with_name(f"{data_path.stem}.return_update.tmp.feather")
nav.to_feather(temp_path)

saved_dataset = ds.dataset(temp_path, format="feather")
if saved_dataset.count_rows() != original_row_count:
    raise RuntimeError("保存后行数发生变化，已停止覆盖正式文件。")
if saved_dataset.schema.names != original_columns:
    raise RuntimeError("保存后列结构发生变化，已停止覆盖正式文件。")

os.replace(temp_path, data_path)
print(f"已更新 {data_path} 的 return 列，共 {original_row_count:,} 行；其他列和行序保持不变。")

已更新 交易日偏股型基金.feather 的 return 列，共 13,266,734 行；其他列和行序保持不变。
